# AFL Coaches Vote Prediction — 2023

This notebook supports the Case Studies in Data Science analysis of whether AFL player match statistics can identify performances recognised by AFL Coaches Association (AFLCA) coaches.

**Datasets**
- AFL Player Match Statistics
- AFLCA Coaches Votes

**Models**
- Logistic Regression
- RBF Support Vector Machine (SVM)

The analysis uses the 2023 AFL season only.


## 1. Setup and load the data

Place the two CSV files in a `data/` folder beside this notebook.  
If they are not found there, the notebook also checks the local `Downloads` folder.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve
)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

stats_path = DATA_DIR / "afl_player_stats.csv"
votes_path = DATA_DIR / "aflca_votes.csv"

# Convenient fallback for local use on macOS
if not stats_path.exists():
    stats_path = Path.home() / "Downloads" / "afl_player_stats.csv"

if not votes_path.exists():
    votes_path = Path.home() / "Downloads" / "aflca_votes.csv"

stats = pd.read_csv(stats_path, low_memory=False)
votes = pd.read_csv(votes_path)

print("Player stats:", stats.shape)
print("AFLCA votes:", votes.shape)


## 2. Preprocess and merge the datasets

The analysis is restricted to 2023. Player names are standardised because the two sources use slightly different naming conventions, including nicknames and middle initials. The cleaned AFLCA votes are then merged onto the player-match statistics and a binary target is created:

- `1` = received at least one AFLCA coaches' vote
- `0` = received no AFLCA coaches' votes


In [ ]:
# Keep only the 2023 season
stats = stats[stats["Year"] == 2023].copy()
votes = votes[votes["Year"] == 2023].copy()

def clean_first_name(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z]", "", regex=True)
    )

def clean_surname(series):
    return (
        series.astype(str)
        .str.strip()
        # Remove a leading middle initial, e.g. "J Lynch" -> "Lynch"
        .str.replace(r"^[A-Za-z]\.?\s+", "", regex=True)
        .str.lower()
        .str.replace(r"[^a-z]", "", regex=True)
    )

stats["first_clean"] = clean_first_name(stats["First.Name"])
stats["surname_clean"] = clean_surname(stats["Surname"])

votes["first_clean"] = clean_first_name(votes["First.Name"])
votes["surname_clean"] = clean_surname(votes["Surname"])

# Known 2023 naming differences between the two data sources
name_fixes = {
    ("joshua", "rachele"): "josh",
    ("nicholas", "newman"): "nic",
    ("cameron", "rayner"): "cam",
    ("joshua", "weddle"): "josh",
    ("william", "rioli"): "junior",
}

for (old_first, surname), new_first in name_fixes.items():
    mask = (
        (votes["first_clean"] == old_first)
        & (votes["surname_clean"] == surname)
    )
    votes.loc[mask, "first_clean"] = new_first

# Keep only rounds represented in the AFLCA dataset
valid_rounds = votes["Round.Number"].unique()
stats = stats[stats["Round.Number"].isin(valid_rounds)].copy()

# Check that every AFLCA vote recipient matches a player-stat record
vote_check = votes.merge(
    stats[
        ["Year", "Round.Number", "first_clean", "surname_clean"]
    ].drop_duplicates(),
    on=["Year", "Round.Number", "first_clean", "surname_clean"],
    how="left",
    indicator=True
)

unmatched = vote_check[vote_check["_merge"] == "left_only"].copy()

print("Total AFLCA vote records:", len(votes))
print("Unmatched vote records:", len(unmatched))

if len(unmatched) > 0:
    display(
        unmatched[
            ["First.Name", "Surname", "Round.Number", "Team.Name", "Coaches.Votes"]
        ]
    )

# Merge coaches' votes onto player-match statistics
vote_data = votes[
    [
        "Year",
        "Round.Number",
        "first_clean",
        "surname_clean",
        "Coaches.Votes",
    ]
].copy()

merged = stats.merge(
    vote_data,
    on=["Year", "Round.Number", "first_clean", "surname_clean"],
    how="left"
)

merged["Coaches.Votes"] = merged["Coaches.Votes"].fillna(0).astype(int)
merged["Received_Vote"] = (merged["Coaches.Votes"] > 0).astype(int)

print("\nMerged rows:", len(merged))
print("\nTarget counts:")
print(merged["Received_Vote"].value_counts())

print("\nTarget percentages:")
print((merged["Received_Vote"].value_counts(normalize=True) * 100).round(2))

clean_path = OUTPUT_DIR / "afl_2023_cleaned.csv"
merged.to_csv(clean_path, index=False)
print("\nSaved cleaned dataset to:", clean_path)


## 3. Select features and create the train/test split

Fourteen performance variables are used. The split is grouped by `Game.ID`, so players from the same match cannot appear in both the training and test sets.


In [ ]:
features = [
    "Goals",
    "Behinds",
    "Disposals",
    "Marks",
    "Tackles",
    "Contested.Possessions",
    "Inside.50s",
    "Total.Clearances",
    "Metres.Gained",
    "Score.Involvements",
    "Intercepts",
    "Goal.Assists",
    "Disposal.Efficiency.Percentage",
    "Time.On.Ground.Percentage",
]

target = "Received_Vote"

missing_features = [f for f in features if f not in merged.columns]
if missing_features:
    raise KeyError(f"Missing feature columns: {missing_features}")

X = merged[features].copy()
y = merged[target].copy()
groups = merged["Game.ID"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print(f"Training positive rate: {y_train.mean():.3f}")
print(f"Testing positive rate:  {y_test.mean():.3f}")


## 4. Logistic Regression

Median imputation and standardisation are fitted inside the pipeline using the training data only. Balanced class weights are used because vote recipients are the minority class.


In [ ]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_pred_lr = logistic_model.predict(X_test)
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_ap = average_precision_score(y_test, y_prob_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("LOGISTIC REGRESSION")
print("-------------------")
print(f"Accuracy:          {lr_accuracy:.3f}")
print(f"Precision:         {lr_precision:.3f}")
print(f"Recall:            {lr_recall:.3f}")
print(f"F1-score:          {lr_f1:.3f}")
print(f"Average Precision: {lr_ap:.3f}")
print("\nConfusion Matrix:")
print(lr_cm)

coefficients = logistic_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": coefficients
})

coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
coef_df.sort_values("Coefficient", ascending=False)


## 5. RBF Support Vector Machine


In [ ]:
svm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)
y_prob_svm = svm_model.predict_proba(X_test)[:, 1]

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_ap = average_precision_score(y_test, y_prob_svm)
svm_cm = confusion_matrix(y_test, y_pred_svm)

print("RBF SVM")
print("-------")
print(f"Accuracy:          {svm_accuracy:.3f}")
print(f"Precision:         {svm_precision:.3f}")
print(f"Recall:            {svm_recall:.3f}")
print(f"F1-score:          {svm_f1:.3f}")
print(f"Average Precision: {svm_ap:.3f}")
print("\nConfusion Matrix:")
print(svm_cm)


## 6. Model comparison


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "RBF SVM"],
    "Accuracy": [lr_accuracy, svm_accuracy],
    "Precision": [lr_precision, svm_precision],
    "Recall": [lr_recall, svm_recall],
    "F1-score": [lr_f1, svm_f1],
    "Average Precision": [lr_ap, svm_ap],
})

comparison.round(3)


## 7. Figure 1 — Precision–Recall curves

The Precision–Recall curve is useful because the positive class is relatively uncommon.


In [ ]:
lr_curve_precision, lr_curve_recall, _ = precision_recall_curve(
    y_test, y_prob_lr
)

svm_curve_precision, svm_curve_recall, _ = precision_recall_curve(
    y_test, y_prob_svm
)

baseline = y_test.mean()

plt.figure(figsize=(8, 6))

plt.plot(
    lr_curve_recall,
    lr_curve_precision,
    linewidth=2,
    label=f"Logistic Regression (AP = {lr_ap:.3f})"
)

plt.plot(
    svm_curve_recall,
    svm_curve_precision,
    linewidth=2,
    label=f"RBF SVM (AP = {svm_ap:.3f})"
)

plt.axhline(
    baseline,
    linestyle="--",
    linewidth=1,
    label=f"Baseline (Positive Rate = {baseline:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves for AFLCA Vote Prediction")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

figure1_path = OUTPUT_DIR / "precision_recall_curve.png"
plt.savefig(figure1_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure1_path)


## 8. Figure 2 — Logistic Regression coefficients

Because the predictors were standardised, the Logistic Regression coefficients can be compared more meaningfully. Positive coefficients indicate a positive association with the probability of receiving coaches' votes, holding the other included predictors constant.


In [ ]:
coef_plot = coef_df.sort_values("Coefficient", ascending=True)

plt.figure(figsize=(9, 7))
plt.barh(coef_plot["Feature"], coef_plot["Coefficient"])
plt.axvline(x=0, linestyle="--", linewidth=1)

plt.xlabel("Standardised Logistic Regression Coefficient")
plt.ylabel("Player Performance Attribute")
plt.title("Performance Attributes Associated with AFLCA Coaches' Votes")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

figure2_path = OUTPUT_DIR / "logistic_regression_coefficients.png"
plt.savefig(figure2_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure2_path)


## 9. Summary

The two models are compared using accuracy, precision, recall, F1-score and average precision. Logistic Regression also provides interpretable coefficients that help identify which recorded performance attributes are most strongly associated with receiving AFLCA coaches' votes.

The final report should use the values produced by the latest complete run of this notebook.
